In [1]:
# Cellule 1 — Imports et configuration

from pathlib import Path
import json

import pandas as pd
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
TARGET = "Bankrupt?"

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "data.csv"
)

SELECTED_FEATURES = [
    "Quick Ratio",
    "ROA(B) before interest and depreciation after tax",
    "Borrowing dependency",
    "Research and development expense rate",
    "Quick Assets/Current Liability",
]

API_FIELD_MAPPING = {
    "Quick Ratio":
        "quick_ratio",

    "ROA(B) before interest and depreciation after tax":
        "roa_before_interest_and_depreciation_after_tax",

    "Borrowing dependency":
        "borrowing_dependency",

    "Research and development expense rate":
        "research_and_development_expense_rate",

    "Quick Assets/Current Liability":
        "quick_assets_current_liability",
}

In [2]:
# Cellule 2 — Chargement et reconstruction du même split train/test

df = pd.read_csv(DATA_PATH)

df.columns = df.columns.str.strip()

X = df[SELECTED_FEATURES]
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

test_df = X_test.copy()
test_df[TARGET] = y_test

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

print("\nFaillites dans le test :", (y_test == 1).sum())
print("Entreprises saines :", (y_test == 0).sum())

Train : (5455, 5)
Test  : (1364, 5)

Faillites dans le test : 44
Entreprises saines : 1320


In [3]:
# Cellule 3 — Sélection de 5 faillites et 5 entreprises saines

bankrupt_samples = (
    test_df[test_df[TARGET] == 1]
    .sample(
        n=5,
        random_state=RANDOM_STATE,
    )
)

healthy_samples = (
    test_df[test_df[TARGET] == 0]
    .sample(
        n=5,
        random_state=RANDOM_STATE,
    )
)

display(bankrupt_samples)
display(healthy_samples)

,Quick Ratio,ROA(B) before interest and depreciation after tax,Borrowing dependency,Research and development expense rate,Quick Assets/Current Liability,Bankrupt?
4492,0.001731,0.344237,0.377836,9.920000e+09,0.001828,1
6412,0.004303,0.508004,0.382711,5.550000e+09,0.005538,1
6499,0.003512,0.385674,0.376844,1.970000e+09,0.003791,1
1279,0.010531,0.540554,0.378551,7.230000e+09,0.010963,1
1311,0.004885,0.555758,0.384483,0.000000e+00,0.004826,1


,Quick Ratio,ROA(B) before interest and depreciation after tax,Borrowing dependency,Research and development expense rate,Quick Assets/Current Liability,Bankrupt?
5259,1.365088e-02,0.611542,0.372200,9.150000e+09,0.013529,0
6605,3.501679e-02,0.572033,0.369637,5.400000e+09,0.035339,0
2336,8.920000e+09,0.563467,0.390902,0.000000e+00,0.000484,0
1065,4.036871e-03,0.553456,0.380938,1.900000e+09,0.004279,0
3907,6.932739e-03,0.512768,0.369637,6.280000e+08,0.007535,0


In [4]:
# Cellule 4 — Format JSON prêt pour l'API

def row_to_api_payload(row):

    return {
        api_name: float(row[dataset_name])
        for dataset_name, api_name
        in API_FIELD_MAPPING.items()
    }


print("===== 5 ENTREPRISES EN FAILLITE =====\n")

for _, row in bankrupt_samples.iterrows():

    payload = row_to_api_payload(row)

    print(
        json.dumps(
            payload,
            ensure_ascii=False,
        )
    )


print("\n===== 5 ENTREPRISES SAINES =====\n")

for _, row in healthy_samples.iterrows():

    payload = row_to_api_payload(row)

    print(
        json.dumps(
            payload,
            ensure_ascii=False,
        )
    )

===== 5 ENTREPRISES EN FAILLITE =====

{"quick_ratio": 0.0017305333710964, "roa_before_interest_and_depreciation_after_tax": 0.344236843514107, "borrowing_dependency": 0.377835882822653, "research_and_development_expense_rate": 9920000000.0, "quick_assets_current_liability": 0.0018276349143365}
{"quick_ratio": 0.0043032513315874, "roa_before_interest_and_depreciation_after_tax": 0.508003640451844, "borrowing_dependency": 0.382710816559789, "research_and_development_expense_rate": 5550000000.0, "quick_assets_current_liability": 0.0055384267420232}
{"quick_ratio": 0.0035115978175566, "roa_before_interest_and_depreciation_after_tax": 0.385673751271481, "borrowing_dependency": 0.376843896994518, "research_and_development_expense_rate": 1970000000.0, "quick_assets_current_liability": 0.0037910115774321}
{"quick_ratio": 0.0105310504100503, "roa_before_interest_and_depreciation_after_tax": 0.540553562824562, "borrowing_dependency": 0.378550515228716, "research_and_development_expense_rate": 7